# S1 · первый классификатор

От признаков привода до двух обученных моделей, конкретной ошибки и воспроизводимого отчёта.


## Данные и форма массивов

Данные синтетические: 24 привода, по 7 полётов, по 24 окна на полёт. Окно длится 20 секунд, шаг равен 10 секундам.

Признак — измеримое описание окна, доступное к моменту решения. $X\in\mathbb R^{N\times d}$ содержит строки окон и столбцы признаков. Метка $y$ повторяет историческое решение по полёту: **1 — в учебном архиве полёту назначен дополнительный осмотр; 0 — назначение отсутствует**.

Для модели выбираются шесть физических признаков. Готовое разбиение: приводы 00–17, полёты 00–04 — train, 05–06 — validation; приводы 18–23 оставлены для test. На S1 этот протокол предоставлен для разбора первой модели.

Три строки таблицы показывают соответствие признаков $X$ и меток $y$.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_sau.course_case import course_holdout_masks, generate_course_case
from ml_sau.first_model import build_preview_report

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "configs" / "first-model.json").is_file()
)
config_path = PROJECT_ROOT / "configs" / "first-model.json"
settings = json.loads(config_path.read_text(encoding="utf-8"))
data_seed = int(settings["data_seed"])
threshold = float(settings["demonstration_threshold"])
case = generate_course_case(seed=data_seed)
train, validation, _ = course_holdout_masks(case)
validation_flight_id = case.flight_id[validation]
validation_rows = np.flatnonzero(validation)
units = ("deg", "A", "deg C", "g", "kN", "V")
np.set_printoptions(precision=3, suppress=True)

raise NotImplementedError("S1 block 1: выбор признаков и меток")

assert features == tuple(settings["features"])
print("Исходная таблица:", case.X.shape, "X:", X.shape, "y:", y.shape)
print("Полётов:", len(all_flights), "Назначен осмотр:", int(flight_labels.sum()))
print("Окон train/validation:", int(train.sum()), int(validation.sum()))
print("Признаки и единицы:", list(zip(features, units)))
for row in (0, 1, 24):
    print(case.flight_id[row], "t=", case.window_start_s[row], "y=", y[row], "X=", X[row])
print("flight_signature первых двух окон:", case.X[:2, -1])


## Модель, обучение и score окна

Логистическая регрессия вычисляет $z=b+\sum_j w_j\widetilde x_j$ и $s=1/(1+e^{-z})$. Стандартизация задаёт $\widetilde x_j=(x_j-\mu_j)/\sigma_j$, где $\mu_j,\sigma_j$ рассчитаны только по train. Физическая единица исходного столбца сохраняется в описании данных.

`fit` оценивает параметры по обучающим данным; `predict_proba` применяет найденные параметры. Столбец класса 1 содержит score — оценку вероятности назначения осмотра по признакам окна.

Конвейер задаётся явно; после обучения проверяются формы входа и выхода.

In [ ]:
raise NotImplementedError("S1 block 2: обучение и score окна")

assert tuple(model.classes_) == (0, 1)
print("train:", X_train.shape, y_train.shape)
print("validation:", X_validation.shape, "predict_proba:", probability_table.shape)
print("Порядок классов:", model.classes_, "score:", window_score.shape)
print("Первые три score:", window_score[:3])
print("Средние train:", model[0].mean_)
print("Коэффициенты при стандартизованных признаках:", model[-1].coef_[0])


## От окон к решению и ошибкам

Score полёта — максимум score его окон. Это заданное правило агрегации: достаточно одного окна выше порога для предложения осмотра. Порог $\tau=0.5$ фиксируется до сравнения моделей.

Accuracy — доля полётов, в которых решение модели совпало с исторической меткой. FP: при метке 0 модель назначила осмотр. FN: при метке 1 модель осмотр не назначила. TP и TN — совпадения для меток 1 и 0 соответственно.

Функция агрегации, правило порога и маски ошибок связывают оценки окон с решениями по полётам. Результат сопоставляется с постоянным ответом 0.

In [ ]:
raise NotImplementedError("S1 block 3: агрегация, порог и ошибки")

print("Полётов validation:", len(flights), "Назначен осмотр:", int(flight_target.sum()))
print("accuracy:", accuracy, "верных решений:", int(np.sum(prediction == flight_target)))
print("TP/TN/FP/FN:", int(np.sum((flight_target == 1) & prediction)),
      int(np.sum((flight_target == 0) & ~prediction)), int(false_positive.sum()), int(false_negative.sum()))
print("Постоянный ответ 0, accuracy:", np.mean(flight_target == 0))
print("FP:", flights[false_positive])
print("FN:", flights[false_negative])


## Повторное обучение без температуры

Меняется один внешний выбор — список признаков. Все параметры сокращённой модели оцениваются заново. Сохраняются полёты train и validation, семейство модели, агрегация, порог и метрика.

Температура содержит сведения о нагреве привода. Сравнение полной и сокращённой моделей показывает, как её исключение изменило прогнозы на выбранных отложенных полётах.

Сопоставляются изменившиеся решения и значения accuracy.

In [ ]:
raise NotImplementedError("S1 block 4: повторное обучение без температуры")

np.testing.assert_array_equal(flights, reduced_flights)
np.testing.assert_array_equal(flight_target, reduced_target)
print("Полный набор:", features, "accuracy:", accuracy)
print("Без температуры:", reduced_features, "accuracy:", reduced_accuracy)
for index in np.flatnonzero(changed):
    print(flights[index], "y=", flight_target[index],
          "scores:", round(flight_score[index], 3), round(reduced_score[index], 3),
          "решения:", int(prediction[index]), int(reduced_prediction[index]))


## Реальная ложная тревога и корректное решение

Сравниваются первый FP и первый TN в таблице полётов. В обоих случаях историческая метка равна 0. Для каждого берётся окно с максимальным score: именно оно определило решение при выбранной агрегации.

По индексам окон сравниваются их физические значения. Разность признаков описывает различие двух найденных окон. Стандартизованные значения позволяют сопоставить таблицу с графиком на слайде.

In [ ]:
raise NotImplementedError("S1 block 5: фактическая ошибка и корректный полёт")

print("FP, TN:", example_flights)
print("Начала окон, с:", case.window_start_s[example_rows])
print("Признак / единица / FP / TN / разность FP - TN")
for index, name in enumerate(features):
    print(name, units[index], *np.round(profile_values[:, index], 3),
          round(profile_values[0, index] - profile_values[1, index], 3))
print("В масштабе train, строки FP и TN:\n", scaled_profiles)
print("Коэффициенты модели:", model[-1].coef_[0])


## Сохранение результата и воспроизводимость

Готовая функция `build_preview_report` получает вычисленные в notebook scores, решения и индексы разобранных окон. Она оформляет их в JSON, не выполняя обучение заново. Результат notebook сохраняется в `reports/s1-notebook.json`.

Отдельный процесс CLI повторяет эксперимент по конфигурации и пишет `reports/s1-first-model.json`. Оба результата сравниваются целиком. Равенство подтверждает воспроизведение расчёта в закреплённом окружении.

In [ ]:
report = build_preview_report(
    case=case, features=features, data_seed=data_seed, threshold=threshold,
    model=model, flights=flights, target=flight_target,
    score=flight_score, prediction=prediction,
    reduced_features=reduced_features, reduced_score=reduced_score,
    reduced_prediction=reduced_prediction, example_rows=example_rows,
)
notebook_report_path = PROJECT_ROOT / "reports" / "s1-notebook.json"
notebook_report_path.parent.mkdir(parents=True, exist_ok=True)
notebook_report_path.write_text(
    json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
cli_report_path = PROJECT_ROOT / "reports" / "s1-first-model.json"
completed = subprocess.run(
    [sys.executable, "-m", "ml_sau.first_model", "--config", str(config_path),
     "--output", str(cli_report_path)],
    cwd=PROJECT_ROOT, check=True, capture_output=True, text=True,
)
cli_report = json.loads(cli_report_path.read_text(encoding="utf-8"))

raise NotImplementedError("S1 block 6: сверка сохранённого результата с CLI")

print("Результат notebook:", notebook_report_path)
print("Независимый запуск CLI:", cli_report_path)
print("Совпали scores, решения двух моделей и окна разобранных полётов.")
